# GLAAM-4X v4 — Unified Multi-Dataset Fundus Disease Detection (Google Colab)

This notebook trains **GLAAM-4X**, a multi-label deep learning classifier for detecting four sight-threatening diseases — Cataract, Diabetic Retinopathy (DR), Glaucoma, and Myopia — from a single retinal fundus photograph. The model is trained on a unified corpus of public datasets plus synthetic cataract images, and is designed to be lightweight enough for screening workflows while retaining clinical-grade discriminative performance.

The four diseases differ markedly in visual appearance and prevalence, making this a challenging multi-label, imbalanced-classification problem. The notebook addresses these challenges through disease-specific attention heads, asymmetric loss, balanced sampling, differential augmentation, and a carefully tuned learning-rate schedule. All training artifacts, including checkpoints, metrics, graphs, and an inference package, are saved to Google Drive so they persist beyond the Colab session.

To use the notebook, upload the project folder (containing `models/`, `utils/`, `raw/`, `synthetic/`, and the four CSV splits) to the Drive path set in `DRIVE_BASE`, then run the cells sequentially.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Section 1: Environment Setup and Dependency Installation
# ═══════════════════════════════════════════════════════════════

import subprocess
import sys

# Install required packages (Colab already has torch, but pin versions for reproducibility)
# CRITICAL: pandas, scipy, and scikit-learn are pre-installed on Colab compiled against
# NumPy 2.x.  We must force-reinstall them together with numpy<2.0 so their binary wheels
# are rebuilt against the NumPy 1.x ABI.  Otherwise `import pandas` triggers:
#   "ValueError: numpy.dtype size changed, may indicate binary incompatibility"
packages = [
    "numpy<2.0",
    "pandas",
    "scipy",
    "scikit-learn",
    "tqdm",
    "albumentations",
    "opencv-python-headless",
    "safetensors",
    "psutil",
]

print("Installing packages (force-reinstall to fix NumPy ABI)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall"] + packages)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

# Verify NumPy / pandas ABI compatibility after reinstall
import numpy as np
import pandas as pd
print(f"\nNumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("✅ NumPy ABI check passed — pandas imported successfully.")

# Section 2: Google Drive Mount and Project Setup

This section mounts Google Drive inside the Colab runtime and verifies the required project files. Because Colab's local disk is temporary, Drive is used for persistent storage of data, code, checkpoints, and results. The `DRIVE_BASE` variable points to the project folder (`/content/drive/MyDrive/Backup/dataset/cataract_detection` by default) and should be updated if your folder path differs.

The verification step checks for `models/` (GLAAM-4X architecture), `utils/` (ASL loss implementation), `raw/` and `synthetic/` (fundus images), and the four CSV splits (`train_v4.csv`, `val_tune_v4.csv`, `test_v4.csv`, `test_v4_extended.csv`). All must be present before proceeding.

In [ ]:
import json
import os
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# Mount Google Drive
# ═══════════════════════════════════════════════════════════════
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("google.colab not available; assuming local environment with DRIVE_BASE set manually")

# ═══════════════════════════════════════════════════════════════
# CONFIGURE YOUR PATHS HERE
# ═══════════════════════════════════════════════════════════════
# Based on your Drive structure:
#   MyDrive/cataract_detection/
#   ├── raw/          ← fundus images
#   ├── synthetic/    ← synthetic cataract images
#   ├── train_v4.csv  ← you need to upload this too
#   ├── val_tune_v4.csv
#   ├── test_v4.csv
#   └── test_v4_extended.csv
#
# You also need to upload from your project:
#   ├── models/       ← glaam_4x.py, etc.
#   └── utils/        ← losses.py, etc.
# ═══════════════════════════════════════════════════════════════
DRIVE_BASE = "/content/drive/MyDrive/Backup/dataset/cataract_detection"  # <-- CHANGE IF YOUR FOLDER NAME IS DIFFERENT

# Add project code to Python path for imports
import sys
sys.path.insert(0, DRIVE_BASE)

DISEASE_NAMES = ['Cataract', 'DR', 'Glaucoma', 'Myopia']

print(f"Drive mounted. Project base: {DRIVE_BASE}")
print(f"Python path added: {DRIVE_BASE}")
print("Disease names:", DISEASE_NAMES)

# Verify project structure exists
print("\nChecking required folders/files:")
for sub in ["models", "utils", "raw", "synthetic"]:
    p = Path(DRIVE_BASE) / sub
    print(f"  {'✅' if p.exists() else '❌'} {sub}/ -> {p}")

# Check for CSVs
csv_files = ["train_v4.csv", "val_tune_v4.csv", "test_v4.csv", "test_v4_extended.csv"]
print("\nChecking CSV files:")
for csv in csv_files:
    p = Path(DRIVE_BASE) / csv
    print(f"  {'✅' if p.exists() else '❌'} {csv} -> {p}")

# Warn about missing files
missing = []
for sub in ["models", "utils", "raw", "synthetic"]:
    if not (Path(DRIVE_BASE) / sub).exists():
        missing.append(sub + "/")
for csv in csv_files:
    if not (Path(DRIVE_BASE) / csv).exists():
        missing.append(csv)

if missing:
    print(f"\n⚠️  WARNING: Missing files/folders: {missing}")
    print("   Please upload them to your Google Drive before running the next cells.")
else:
    print("\n✅ All required files found!")


# Section 3: Cached Data Loading and Multi-Dataset CSV Exploration

This section loads the dataset onto the local Colab SSD (`/tmp/data`) using a three-tier caching strategy to minimize the 70–90 minute copy time on subsequent sessions:

1. **Same-session restart**: if the data is already on local SSD (marked by `.cache_complete`), loading is skipped entirely.
2. **Cross-session cache**: if a `dataset_cache.tar` archive exists on Drive, it is extracted to local SSD in ~10–15 minutes — much faster than copying thousands of small files through Drive's FUSE filesystem.
3. **First run**: the dataset is copied from Drive to local SSD (~70–90 min), then a `dataset_cache.tar` is created on Drive (~30–40 min one-time cost) so all future sessions use the fast extraction path.

After loading, the four CSV splits are read with pandas. Each CSV contains one row per image with an `image_path` and four binary disease labels: `Cataract`, `DR`, `Glaucoma`, and `Myopia`.

**Splits:**
- **Train**: 22,281 images — used for model updates.
- **Val tune**: 3,534 images — used for threshold selection and overfitting monitoring.
- **Test v4**: 1,407 images — held-out set for direct comparison with the v3 model.
- **Test v4 extended**: 5,339 images — larger held-out set for publication results.

Class weights are computed from training-set frequencies and displayed for reference; the main imbalance handling is performed by ASL loss and balanced sampling.

In [ ]:
import shutil
import time
import pandas as pd
import numpy as np
import tarfile

# ═══════════════════════════════════════════════════════════════
# Cached dataset loading — avoids 70-90 min re-copy on every session
# ═══════════════════════════════════════════════════════════════
# Strategy:
#   1. If data already on local SSD (same session) → skip
#   2. If tar cache exists on Drive → extract (fast, ~10-15 min)
#   3. First time → copy from Drive, then build tar cache for future
# ═══════════════════════════════════════════════════════════════

local_data_dir = Path("/tmp/data")
cache_tar = Path(DRIVE_BASE) / "dataset_cache.tar"       # single-file archive on Drive
cache_marker = local_data_dir / ".cache_complete"         # marks local SSD as ready

img_root = str(local_data_dir)
csv_root = str(local_data_dir)

def _count_files(directory):
    """Count total files in directory (for verification)."""
    return sum(1 for _ in directory.rglob("*") if _.is_file())

def _verify_local_cache():
    """Check that local SSD has the essential folders and CSVs."""
    for sub in ["raw", "synthetic"]:
        if not (local_data_dir / sub).exists():
            return False
    for csv_name in ["train_v4.csv", "val_tune_v4.csv", "test_v4.csv"]:
        if not (local_data_dir / csv_name).exists():
            return False
    return True

# ─────────────────────────────────────────────────────────────
# Step 1: Check if data already on local SSD (same-session restart)
# ─────────────────────────────────────────────────────────────
if cache_marker.exists() and _verify_local_cache():
    print("✅ Dataset already on local SSD — skipping copy/extract")
    n_files = _count_files(local_data_dir)
    print(f"   {n_files} files found at {local_data_dir}")

# ─────────────────────────────────────────────────────────────
# Step 2: Check if tar cache exists on Drive (cross-session cache)
# ─────────────────────────────────────────────────────────────
elif cache_tar.exists():
    cache_size_gb = cache_tar.stat().st_size / (1024**3)
    print(f"📦 Found dataset cache on Drive: {cache_tar}")
    print(f"   Size: {cache_size_gb:.1f} GB")
    print(f"   Extracting to local SSD (much faster than copying individual files)...")
    t0 = time.time()
    local_data_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(cache_tar, "r") as tar:
        # Extract with progress (tarfile doesn't have built-in progress,
        # but extraction of a single large file is I/O-bound and fast)
        tar.extractall(path=local_data_dir)
    extract_time = time.time() - t0
    cache_marker.touch()
    n_files = _count_files(local_data_dir)
    print(f"✅ Cache extracted in {extract_time:.0f}s ({extract_time/60:.1f} min)")
    print(f"   {n_files} files extracted to {local_data_dir}")

# ─────────────────────────────────────────────────────────────
# Step 3: No cache — copy from Drive, then build tar cache
# ─────────────────────────────────────────────────────────────
else:
    print("No cache found. Copying dataset from Drive to local SSD...")
    print("⚠️  First-time setup: ~70-90 min copy + ~30-40 min cache creation")
    print("   Future sessions will load in ~10-15 min using the cache.\n")

    # Copy from Drive to local SSD
    t0 = time.time()
    drive_data_dir = Path(DRIVE_BASE)
    shutil.copytree(drive_data_dir, local_data_dir, dirs_exist_ok=True)
    copy_time = time.time() - t0
    print(f"✅ Dataset copied in {copy_time:.0f}s ({copy_time/60:.1f} min)")

    n_files = _count_files(local_data_dir)
    print(f"   {n_files} files at {local_data_dir}")

    # Build tar cache on Drive for future sessions
    print(f"\nCreating cache archive on Drive for future sessions...")
    print(f"   This is a one-time cost. Future runs will skip this step.")
    t0 = time.time()
    with tarfile.open(cache_tar, "w") as tar:
        for item in sorted(local_data_dir.iterdir()):
            if item.name == ".cache_complete":
                continue
            print(f"   Adding {item.name}...", end=" ", flush=True)
            tar.add(item, arcname=item.name)
            print("done")
    cache_time = time.time() - t0
    cache_size_gb = cache_tar.stat().st_size / (1024**3)
    print(f"✅ Cache created in {cache_time:.0f}s ({cache_time/60:.1f} min)")
    print(f"   Location: {cache_tar}")
    print(f"   Size: {cache_size_gb:.1f} GB")
    print(f"\n💡 Future Colab sessions will extract this cache in ~10-15 min")
    print(f"   instead of copying for 70-90 min.")

    cache_marker.touch()

print(f"\nCSV root: {csv_root}")
print(f"Image root: {img_root}")

# Load CSVs (they should be in the same folder as raw/ and synthetic/)
train_csv = Path(csv_root) / "train_v4.csv"
val_tune_csv = Path(csv_root) / "val_tune_v4.csv"
test_csv = Path(csv_root) / "test_v4.csv"
test_extended_csv = Path(csv_root) / "test_v4_extended.csv"

if not train_csv.exists():
    raise FileNotFoundError(
        f"Train CSV not found: {train_csv}\n"
        f"Please upload train_v4.csv, val_tune_v4.csv, test_v4.csv, and test_v4_extended.csv\n"
        f"to your Google Drive folder: {DRIVE_BASE}/"
    )

train_df = pd.read_csv(train_csv)
val_tune_df = pd.read_csv(val_tune_csv)
test_df = pd.read_csv(test_csv)

print(f"Train: {len(train_df)} images")
print(f"Val tune: {len(val_tune_df)} images")
print(f"Test (v3 comparison): {len(test_df)} images")
print(f"\nTrain disease distribution:\n{train_df[DISEASE_NAMES].sum().to_dict()}")
print(f"\nVal disease distribution:\n{val_tune_df[DISEASE_NAMES].sum().to_dict()}")

print(f"\nTest disease distribution:\n{test_df[DISEASE_NAMES].sum().to_dict()}")

# Compute class weights
pos_counts = train_df[DISEASE_NAMES].sum().values.astype(np.float32)
class_weights = torch.tensor(len(train_df) / (pos_counts + 1e-6), dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * 4  # normalize to mean=1
print(f"\nClass weights: {dict(zip(DISEASE_NAMES, class_weights.tolist()))}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CPU & Hardware Info — check available resources before training
# ═══════════════════════════════════════════════════════════════
import os
import multiprocessing
import psutil

print("=" * 50)
print("  HARDWARE RESOURCES")
print("=" * 50)

# CPU info
num_cpus = multiprocessing.cpu_count()
print(f"  CPU cores (logical): {num_cpus}")
print(f"  CPU cores (physical): {psutil.cpu_count(logical=False)}")

# Memory info
mem = psutil.virtual_memory()
mem_total_gb = mem.total / (1024**3)
mem_avail_gb = mem.available / (1024**3)
print(f"  RAM total:     {mem_total_gb:.1f} GB")
print(f"  RAM available:  {mem_avail_gb:.1f} GB")

# Disk info for /tmp (local SSD)
disk = psutil.disk_usage('/tmp')
disk_total_gb = disk.total / (1024**3)
disk_free_gb = disk.free / (1024**3)
print(f"  /tmp SSD total: {disk_total_gb:.1f} GB")
print(f"  /tmp SSD free:  {disk_free_gb:.1f} GB")

# GPU info
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"  GPU:            {gpu_name}")
    print(f"  GPU VRAM:       {gpu_mem_gb:.1f} GB")
else:
    print(f"  GPU:            ❌ Not available (CPU-only mode)")

# Recommended num_workers for DataLoader
recommended_workers = min(4, num_cpus - 1)
print(f"\n  💡 Recommended DataLoader num_workers: {recommended_workers}")
print(f"     (Auto-adjusted in Section 5 based on available CPU cores)")

print("=" * 50)

# Section 4: Differential Augmentation and Dataset Class

This section defines the image transformation pipeline and the `UnifiedDataset` class. Medical image datasets are typically small and expensive to annotate, so augmentation is used to expand the effective training set and improve robustness to rotation, brightness/contrast differences, and minor translation or scale variations across clinics.

Two augmentation strategies are used:
- **Base transform** (`get_transforms`): applied to every training image. Includes flips, rotation, affine warping, brightness/contrast adjustment, Gaussian noise, blur, elastic deformation, grid distortion, and coarse dropout.
- **Strong transform** (`strong_aug`): applied with 70% probability to images containing at least one disease. It uses more aggressive distortion, noise, and dropout to prevent overfitting on the limited number of positive samples.

`UnifiedDataset` loads images with OpenCV, resolves relative paths against both `img_dir/` and `img_dir/raw/`, applies the appropriate augmentation, and returns a normalized image tensor with a 4-dimensional label vector.

In [ ]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset

# Global config placeholder for transforms
IMG_SIZE = 384

def get_transforms(img_size, is_train, minority_aug_prob=0.7):
    _ = minority_aug_prob  # reserved for future differential transform selection
    base = [
        A.Resize(img_size, img_size),
        A.ToFloat(max_value=255),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    if not is_train:
        return A.Compose(base)

    strong = [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.Affine(translate_percent={'x': 0.05, 'y': 0.05}, scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.GaussNoise(std_range=(0.04, 0.2), p=0.3),
        A.GaussianBlur(blur_limit=3, p=0.2),
        A.ElasticTransform(alpha=1, sigma=50, p=0.1),
        A.GridDistortion(p=0.1),
        A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), p=0.2),
    ]
    return A.Compose(strong + base)


class UnifiedDataset(Dataset):
    def __init__(self, df, img_dir, disease_cols, img_size, is_train=False, minority_aug_prob=0.7):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.disease_cols = disease_cols
        self.is_train = is_train
        self.minority_aug_prob = minority_aug_prob
        self.transform = get_transforms(img_size, is_train, minority_aug_prob)
        # Extra strong augmentation for minority-class samples
        self.strong_aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.Affine(translate_percent={'x': 0.1, 'y': 0.1}, scale=(0.8, 1.2), rotate=(-30, 30), p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
            A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=15, p=0.4),
            A.GaussNoise(std_range=(0.08, 0.3), p=0.4),
            A.GaussianBlur(blur_limit=5, p=0.3),
            A.ElasticTransform(alpha=2, sigma=50, p=0.2),
            A.GridDistortion(p=0.2),
            A.CoarseDropout(num_holes_range=(2, 6), hole_height_range=(16, 48), hole_width_range=(16, 48), p=0.3),
            A.Resize(img_size, img_size),
            A.ToFloat(max_value=255),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        if not os.path.isabs(path):
            candidates = [
                os.path.join(self.img_dir, path),
                os.path.join(self.img_dir, "raw", path),
            ]
            for p in candidates:
                if os.path.exists(p):
                    path = p
                    break
            else:
                path = candidates[0]
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        img = self._load_image(img_path)
        labels = row[self.disease_cols].values.astype(np.float32)
        # Differential augmentation: stronger aug for minority-class samples
        if self.is_train and labels.sum() > 0 and np.random.rand() < self.minority_aug_prob:
            augmented = self.strong_aug(image=img)
        else:
            augmented = self.transform(image=img)
        img = augmented['image']
        return img, torch.tensor(labels)


# Instantiate datasets
train_dataset = UnifiedDataset(train_df, img_root, DISEASE_NAMES, IMG_SIZE, is_train=True, minority_aug_prob=0.7)
val_tune_dataset = UnifiedDataset(val_tune_df, img_root, DISEASE_NAMES, IMG_SIZE, is_train=False)
test_dataset = UnifiedDataset(test_df, img_root, DISEASE_NAMES, IMG_SIZE, is_train=False)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val tune dataset: {len(val_tune_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")


# Section 5: Balanced Sampling with WeightedRandomSampler

This section builds the PyTorch DataLoaders. Random sampling would underrepresent rare diseases in every batch because the dataset is heavily skewed toward healthy images. `WeightedRandomSampler` assigns higher sampling probability to images containing diseases, rebalancing the effective training distribution without discarding healthy samples.

The DataLoader configuration uses `num_workers` auto-adjusted to the available CPU cores, `pin_memory=True`, `persistent_workers=True`, and `prefetch_factor=2` to keep the GPU fed during training and avoid CPU bottlenecks.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import multiprocessing

BATCH_SIZE = 32

# Adjust num_workers for Colab's limited CPU (2 logical cores → use 1 to avoid contention)
NUM_WORKERS = min(2, multiprocessing.cpu_count() - 1)
print(f"Using num_workers={NUM_WORKERS} (CPU has {multiprocessing.cpu_count()} logical cores)")

def get_sample_weights(df, disease_cols):
    pos_counts = df[disease_cols].sum().values
    neg_counts = len(df) - pos_counts
    pos_weights = np.sqrt(1.0 / (pos_counts + 1e-6))
    neg_weights = np.sqrt(1.0 / (neg_counts + 1e-6))
    pos_weights = pos_weights / pos_weights.sum()
    neg_weights = neg_weights / neg_weights.sum()
    weights = np.zeros(len(df))
    for i, row in df.iterrows():
        w = 0.0
        for j, col in enumerate(disease_cols):
            w += pos_weights[j] if row[col] == 1 else neg_weights[j]
        weights[i] = w / len(disease_cols)
    return weights

sample_weights = get_sample_weights(train_df, DISEASE_NAMES)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_df) * 2,
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    drop_last=True,
)
val_tune_loader = DataLoader(
    val_tune_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val tune loader: {len(val_tune_loader)} batches")
print(f"Test loader: {len(test_loader)} batches")


# Section 6: GLAAM-4X Model Architecture and Reordering

This section imports the GLAAM-4X backbone and wraps it in a classifier that produces logits in the target disease order.

**GLAAM-4X** (Gated Lightweight Attention Module with 4 disease-specific heads) consists of:
- A **MobileNetV2** backbone pretrained on ImageNet, chosen for low latency and modest memory use while retaining sufficient representational capacity.
- Four disease-specific attention heads that focus on clinically relevant regions:
  - **Cataract**: lens opacity patterns.
  - **DR**: microaneurysms, hemorrhages, and exudates.
  - **Glaucoma**: optic disc and cup morphology.
  - **Myopia**: tessellation and peripapillary atrophy.

The backbone originally outputs logits in the order `[DR, Glaucoma, Cataract, Myopia]`. The wrapper applies `REORDER_IDX = [2, 0, 1, 3]` to align with the CSV order `[Cataract, DR, Glaucoma, Myopia]`. `torch.compile(mode='reduce-overhead')` is applied when available to reduce training overhead.

In [ ]:
import torch.nn as nn

# Import GLAAM_4X from project code on Drive
from models.glaam_4x import GLAAM_4X

class GLAAM4XClassifier(nn.Module):
    REORDER_IDX = [2, 0, 1, 3]  # [DR, Glaucoma, Cataract, Myopia] -> [Cataract, DR, Glaucoma, Myopia]
    def __init__(self, dropout_rate=0.3, pretrained=True):
        super().__init__()
        self.backbone = GLAAM_4X(pretrained=pretrained, dropout_rate=dropout_rate)
    def forward(self, x):
        out = self.backbone(x)
        logits = out['logits']
        return logits[:, self.REORDER_IDX]

# Build model
DROPOUT_RATE = 0.3
model = GLAAM4XClassifier(dropout_rate=DROPOUT_RATE, pretrained=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Sanity check: verify disease order alignment
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    out_logits = model(dummy)
    print(f"Logits shape: {out_logits.shape} (should be [2, 4])")
    print("Expected disease order after reorder:", DISEASE_NAMES)

# PyTorch 2.0+ optimization for ~20-40% speedup
if hasattr(torch, 'compile'):
    print("Compiling model with torch.compile(mode='reduce-overhead')...")
    model = torch.compile(model, mode="reduce-overhead")
else:
    print("torch.compile() not available (PyTorch < 2.0)")


# Section 7: Asymmetric Loss and Optimizer Configuration

This section configures the training objective and optimizer.

**Asymmetric Loss (ASL)** is used because fundus datasets are multi-label and highly imbalanced. ASL down-weights easy negative gradients and applies a small probability clip (`clip=0.05`) to prevent the model from becoming overconfident on the dominant negative class. The hyperparameters `γ_neg=4.0` and `γ_pos=0.0` were selected to focus learning on the rare positive examples.

**AdamW** is the optimizer with `lr=2e-5` and `weight_decay=5e-4`. AdamW decouples weight decay from gradient updates, which generally improves generalization compared to standard Adam.

In [ ]:
from torch.optim import AdamW
from utils.losses import AsymmetricLossOptimized

# Config values
ASL_GAMMA_NEG = 4.0
ASL_GAMMA_POS = 0.0
ASL_CLIP = 0.05
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 5e-4

criterion = AsymmetricLossOptimized(
    gamma_neg=ASL_GAMMA_NEG,
    gamma_pos=ASL_GAMMA_POS,
    clip=ASL_CLIP,
)
print(f"Using AsymmetricLoss (γ_neg={ASL_GAMMA_NEG}, γ_pos={ASL_GAMMA_POS}, clip={ASL_CLIP})")

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"AdamW optimizer: LR={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}")


# Section 8: Warmup and Cosine Annealing Scheduler

This section defines the learning-rate schedule and plots it. A 10-epoch linear warmup gradually raises the learning rate from 0 to the target value, which stabilizes early training when gradients are noisy. After warmup, cosine annealing smoothly reduces the learning rate over the remaining epochs, allowing broad exploration early on and fine convergence near the end. The plot provides a quick sanity check before the full training loop begins.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR
import matplotlib.pyplot as plt

WARMUP_EPOCHS = 10
TOTAL_EPOCHS = 60

def lr_lambda(epoch):
    # Epoch is 0-indexed in LambdaLR
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)  # linearly from 0 to 1
    else:
        # Cosine annealing from 1 to 0 over remaining epochs
        progress = (epoch - WARMUP_EPOCHS) / max(1, TOTAL_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
print(f"Using warmup ({WARMUP_EPOCHS} epochs) + cosine annealing scheduler.")

# Plot learning rate curve
lrs = [LEARNING_RATE * lr_lambda(e) for e in range(TOTAL_EPOCHS)]
plt.figure(figsize=(10, 4))
plt.plot(range(1, TOTAL_EPOCHS + 1), lrs)
plt.axvline(WARMUP_EPOCHS, color='r', linestyle='--', label='Warmup end')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule: Warmup + Cosine Annealing')
plt.legend()
plt.grid(True)
plt.show()


# Section 9: Training Loop with AMP and Gradient Clipping

This section defines `train_epoch()` and `evaluate()`, the core training and validation functions.

**Automatic Mixed Precision (AMP)** runs most forward/backward operations in float16 while keeping loss accumulation in float32. This reduces GPU memory and increases throughput without sacrificing numerical stability. **Gradient clipping** (`max_norm=10.0`) is applied because ASL can generate large gradients on hard negatives, and clipping prevents occasional gradient spikes. Batches that produce non-finite loss are skipped rather than updating the weights.

`evaluate()` runs the model in inference mode to collect logits and labels across the validation or test set; these are used for metrics and threshold optimization.

In [ ]:
from tqdm import tqdm

scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

def train_epoch(model, loader, criterion, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    all_logits, all_labels = [], []
    for batch_idx, (images, labels) in enumerate(tqdm(loader, desc="Training")):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        # Automatic Mixed Precision (AMP)
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            
            # Safety check: detect NaN/Inf loss before backward
            if not torch.isfinite(loss):
                print(f"\n🚨 WARNING: Non-finite loss ({loss.item():.2f}) at batch {batch_idx}")
                print("  Skipping batch...")
                scaler.update()
                continue
            
            scaler.scale(loss).backward()
            # Gradient clipping to prevent explosion from ASL
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            
            # Safety check: detect NaN/Inf loss before backward
            if not torch.isfinite(loss):
                print(f"\n🚨 WARNING: Non-finite loss ({loss.item():.2f}) at batch {batch_idx}")
                print("  Skipping batch...")
                continue
            
            loss.backward()
            # Gradient clipping to prevent explosion from ASL
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
        
        total_loss += loss.item()
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.cpu())
    scheduler.step()  # step after each epoch
    avg_loss = total_loss / len(loader)
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return avg_loss, all_logits, all_labels


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="Evaluating"):
        images = images.to(device)
        logits = model(images)
        all_logits.append(logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy()

print("Training and evaluation functions defined.")


# Section 10: Evaluation Metrics and Threshold Optimization

This section defines `compute_metrics()` and `find_optimal_thresholds()`. For medical screening, the relevant metrics are **AUC** (ranking quality independent of threshold), **F1** (balance between precision and recall on imbalanced classes), **precision** (fraction of predicted positives that are true disease cases), and **recall** (fraction of true disease cases that are detected). A single 0.5 threshold is rarely optimal across diseases with different prevalences, so thresholds are grid-searched from 0.05 to 0.95 on the validation set to maximize per-disease F1. **Macro F1**, the average of the four per-disease F1 scores, is the primary comparison metric because it gives equal weight to each disease regardless of prevalence.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

def compute_metrics(logits, labels, thresholds=None):
    probs = 1 / (1 + np.exp(-logits))
    if thresholds is None:
        thresholds = {d: 0.5 for d in DISEASE_NAMES}
    metrics = {}
    for i, disease in enumerate(DISEASE_NAMES):
        y_true = labels[:, i]
        y_prob = probs[:, i]
        thr = thresholds.get(disease, 0.5)
        y_pred = (y_prob >= thr).astype(int)
        metrics[disease] = {
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5,
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
        }
    metrics['macro_f1'] = np.mean([m['f1'] for m in metrics.values()])
    # probs returned for optional downstream use; silence unused-variable warnings
    _ = probs
    return metrics, probs


def find_optimal_thresholds(logits, labels):
    probs = 1 / (1 + np.exp(-logits))
    thresholds = {}
    for i, disease in enumerate(DISEASE_NAMES):
        y_true = labels[:, i]
        y_prob = probs[:, i]
        best_f1, best_thr = 0, 0.5
        for thr in np.arange(0.05, 0.95, 0.01):
            y_pred = (y_prob >= thr).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[disease] = float(best_thr)
    return thresholds

print("compute_metrics and find_optimal_thresholds defined.")


# Section 11: Test Set Evaluation and Results Aggregation

This section defines `run_test_evaluation()`, which loads the best checkpoint and evaluates it on the held-out test sets. Keeping evaluation in a separate function makes it reusable after training or from another notebook without re-running the full loop.

Two test sets are used:
- **`test_v4`** (1,407 images): same distribution as the previous v3 model, enabling direct comparison.
- **`test_v4_extended`** (5,339 images): a larger, more diverse held-out set for publication-grade results.

The function writes `results.json`, containing the best epoch, best validation macro F1, optimal thresholds, and per-disease AUC, F1, precision, and recall for each test set. This file is the main artifact for reporting and downstream analysis.

In [ ]:
def run_test_evaluation(model, run_dir, test_csv, test_loader_obj, test_extended_csv=None):
    """Evaluate best model on test sets and save results."""
    print("\nLoading best model for test evaluation...")
    best_checkpoint = torch.load(f"{run_dir}/best_model.pth", map_location=device, weights_only=False)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    model.eval()

    final_thresholds = best_checkpoint['optimal_thresholds']

    # Evaluate on both test sets
    test_configs = [("test_v4", test_csv, test_loader_obj)]

    # Add extended test set if available
    if test_extended_csv and test_extended_csv.exists():
        test_extended_df = pd.read_csv(test_extended_csv)
        test_extended_dataset = UnifiedDataset(test_extended_df, img_root, DISEASE_NAMES, IMG_SIZE, is_train=False)
        test_extended_loader = DataLoader(
            test_extended_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )
        test_configs.append(("test_v4_extended", test_extended_csv, test_extended_loader))
        print(f"Extended test set: {len(test_extended_df)} images")
    else:
        print(f"Extended test CSV not found or not provided, skipping")

    all_test_results = {}

    for test_name, test_csv_path, test_loader_obj in test_configs:
        print(f"\n{'─'*40}")
        print(f"  {test_name}")
        print(f"{'─'*40}")

        test_logits, test_labels = evaluate(model, test_loader_obj)
        test_metrics, _ = compute_metrics(test_logits, test_labels, final_thresholds)

        print(f"\n  {test_name} — {len(pd.read_csv(test_csv_path))} images")
        print(f"  Macro F1: {test_metrics['macro_f1']:.4f}")
        for d in DISEASE_NAMES:
            m = test_metrics[d]
            print(f"    {d:12s} AUC={m['auc']:.4f} F1={m['f1']:.4f} P={m['precision']:.4f} R={m['recall']:.4f}")

        all_test_results[test_name] = {
            'num_images': len(pd.read_csv(test_csv_path)),
            'metrics': test_metrics,
        }

    # Save results
    results = {
        'best_epoch': best_checkpoint.get('epoch', 0),
        'best_val_tune_f1': best_checkpoint.get('best_val_f1', 0.0),
        'optimal_thresholds': final_thresholds,
        'test_results': all_test_results,
    }
    with open(f"{run_dir}/results.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)

    print(f"\nTraining complete. Results saved to {run_dir}/")
    return results

print("run_test_evaluation defined.")


# Section 12: Checkpoint Saving, SafeTensor Export, and Full Training Loop

This section runs the complete training loop, including checkpoint initialization/resumption, per-epoch validation, and model selection. Every epoch saves `latest_model.pth` (for resuming after a Colab disconnect), `best_model.pth` (the checkpoint with the highest validation macro F1), `best_model.safetensors` (a pickle-free SafeTensors copy), `config.json` (hyperparameters), and `training_history.json` (per-epoch metrics). Resume support is essential for Colab, where sessions can time out; if `latest_model.pth` exists, training continues from that epoch. Early stopping halts training after 10 epochs without validation-macro-F1 improvement to prevent overfitting and save compute. All artifacts are written directly to Google Drive because Colab's local disk is ephemeral.

In [ ]:
try:
    import safetensors.torch as _safetensors_torch
    save_file = _safetensors_torch.save_file
except Exception:  # pragma: no cover
    _safetensors_torch = None  # type: ignore
    save_file = None

# ═══════════════════════════════════════════════════════════════
# Training configuration — checkpoints saved to Drive
# ═══════════════════════════════════════════════════════════════
MODEL_NAME = "glaam4x_unified_v4_asl_384"
RUN_DIR = f"{DRIVE_BASE}/checkpoints/{MODEL_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)

with open(f"{RUN_DIR}/config.json", 'w') as f:
    json.dump({
        "model_name": MODEL_NAME,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "dropout_rate": DROPOUT_RATE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "epochs": TOTAL_EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "asl_gamma_neg": ASL_GAMMA_NEG,
        "asl_gamma_pos": ASL_GAMMA_POS,
        "asl_clip": ASL_CLIP,
    }, f, indent=2)

# Resume support
start_epoch = 1
best_val_f1 = 0.0
best_epoch = 0
patience = 10
patience_counter = 0
latest_ckpt_path = os.path.join(RUN_DIR, "latest_model.pth")

# Training history tracker
history = {
    'epoch': [], 'lr': [],
    'train_loss': [], 'train_macro_f1': [],
    'val_loss': [], 'val_macro_f1': [], 'val_macro_f1_opt': [],
    'val_auc': {d: [] for d in DISEASE_NAMES},
    'val_f1': {d: [] for d in DISEASE_NAMES},
    'val_precision': {d: [] for d in DISEASE_NAMES},
    'val_recall': {d: [] for d in DISEASE_NAMES},
}

if os.path.exists(latest_ckpt_path):
    print(f"Found checkpoint {latest_ckpt_path}, resuming...")
    ckpt = torch.load(latest_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt.get('epoch', 0) + 1
    best_val_f1 = ckpt.get('best_val_f1', 0.0)
    patience_counter = ckpt.get('patience_counter', 0)
    scheduler = LambdaLR(optimizer, lr_lambda, last_epoch=start_epoch - 1)
    # Restore history if available
    if 'history' in ckpt:
        history = ckpt['history']
    print(f"Resumed at epoch {start_epoch} | Best val F1 so far: {best_val_f1:.4f}")
else:
    print("No checkpoint found, starting from scratch.")

# ═══════════════════════════════════════════════════════════════
# Full Training Loop
# ═══════════════════════════════════════════════════════════════
print("\nStarting training...")
for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    print(f"\n{'='*60}")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch}/{TOTAL_EPOCHS} | LR: {current_lr:.2e}")
    print(f"{'='*60}")

    train_loss, train_logits, train_labels = train_epoch(model, train_loader, criterion, optimizer, scheduler)
    train_metrics, _ = compute_metrics(train_logits, train_labels)

    # Validation on tuning set
    val_logits, val_labels = evaluate(model, val_tune_loader)
    val_metrics, _ = compute_metrics(val_logits, val_labels)
    optimal_thresholds = find_optimal_thresholds(val_logits, val_labels)
    val_metrics_opt, _ = compute_metrics(val_logits, val_labels, optimal_thresholds)

    # Compute validation loss for plotting
    val_loss = criterion(
        torch.tensor(val_logits, dtype=torch.float32),
        torch.tensor(val_labels, dtype=torch.float32),
    ).item()

    print(f"Train Loss: {train_loss:.4f} | Val Tune Macro F1: {val_metrics['macro_f1']:.4f} (thr=0.5)")
    print(f"Val Tune Macro F1 (optimal thr): {val_metrics_opt['macro_f1']:.4f}")
    for d in DISEASE_NAMES:
        m = val_metrics_opt[d]
        print(f"  {d:12s} AUC={m['auc']:.4f} F1={m['f1']:.4f} P={m['precision']:.4f} R={m['recall']:.4f} thr={optimal_thresholds[d]:.2f}")

    # Record history
    history['epoch'].append(epoch)
    history['lr'].append(current_lr)
    history['train_loss'].append(train_loss)
    history['train_macro_f1'].append(train_metrics['macro_f1'])
    history['val_loss'].append(val_loss)  # Now computed from val logits
    history['val_macro_f1'].append(val_metrics['macro_f1'])
    history['val_macro_f1_opt'].append(val_metrics_opt['macro_f1'])
    for d in DISEASE_NAMES:
        history['val_auc'][d].append(val_metrics_opt[d]['auc'])
        history['val_f1'][d].append(val_metrics_opt[d]['f1'])
        history['val_precision'][d].append(val_metrics_opt[d]['precision'])
        history['val_recall'][d].append(val_metrics_opt[d]['recall'])

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_metrics_opt': val_metrics_opt,
        'optimal_thresholds': optimal_thresholds,
        'best_val_f1': best_val_f1,
        'patience_counter': patience_counter,
        'history': history,
    }

    if val_metrics_opt['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics_opt['macro_f1']
        best_epoch = epoch
        patience_counter = 0
        torch.save(checkpoint, f"{RUN_DIR}/best_model.pth")
        if save_file is not None:
            save_file(model.state_dict(), f"{RUN_DIR}/best_model.safetensors")
            print(f"New best model saved (tune macro F1 = {best_val_f1:.4f})")
        else:
            print(f"New best model saved (tune macro F1 = {best_val_f1:.4f}) — safetensors unavailable")
    else:
        patience_counter += 1

    torch.save(checkpoint, f"{RUN_DIR}/latest_model.pth")

    if patience_counter >= patience:
        print(f"Early stopping triggered after {patience} epochs without improvement")
        break


print(f"\nTraining complete. Best val F1: {best_val_f1:.4f} at epoch {best_epoch}")

# Save history to Drive for later plotting
import json as _json
history_path = f"{RUN_DIR}/training_history.json"
with open(history_path, 'w') as f:
    _json.dump(history, f, indent=2)
print(f"Training history saved to {history_path}")


# Section 12b: Training Visualization Graphs

This section generates publication-quality plots from the saved training history and the best-epoch validation predictions. The eleven figures are: training loss, macro F1 progression (train vs. validation with fixed and optimal thresholds), per-disease validation AUC, per-disease validation F1, per-disease precision-recall trends, the actual learning-rate schedule, a combined dashboard, train-vs-validation loss (for overfitting detection), per-disease ROC curves, per-disease precision-recall curves, and per-disease confusion matrices at optimal thresholds. Together they show whether the model is learning, whether overfitting is occurring, how each disease's discriminative and calibrated performance evolves, and the final false-positive/false-negative trade-offs. All figures are saved at 300 DPI under `checkpoints/glaam4x_unified_v4_asl_384/training_graphs/`.

In [ ]:
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════════
# Publication-quality training graphs
# ═══════════════════════════════════════════════════════════════

GRAPHS_DIR = f"{RUN_DIR}/training_graphs"
os.makedirs(GRAPHS_DIR, exist_ok=True)

# Style settings for publication
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'legend.fontsize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

epochs = history['epoch']
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']  # Blue, Orange, Green, Purple

# ─────────────────────────────────────────────────────────────
# Graph 1: Loss Curve (Train + Validation)
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss', alpha=0.8)
ax.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Validation Loss', alpha=0.8)
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('ASL Loss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(1, max(epochs))
fig.savefig(f"{GRAPHS_DIR}/01_loss_curve.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 2: Macro F1 Progression
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, history['val_macro_f1'], 'orange', linewidth=2, label='Val F1 (thr=0.5)', alpha=0.7)
ax.plot(epochs, history['val_macro_f1_opt'], 'green', linewidth=2, label='Val F1 (optimal thr)')
ax.plot(epochs, history['train_macro_f1'], 'blue', linewidth=1.5, label='Train F1', alpha=0.5)
ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best ({best_epoch})')
ax.axhline(best_val_f1, color='green', linestyle=':', alpha=0.5, label=f'Best F1={best_val_f1:.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Macro F1 Score')
ax.set_title('Macro F1 Score Progression')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(1, max(epochs))
fig.savefig(f"{GRAPHS_DIR}/02_macro_f1_progression.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 3: Per-Disease AUC Curves
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for i, d in enumerate(DISEASE_NAMES):
    ax.plot(epochs, history['val_auc'][d], linewidth=2, color=colors[i], label=d)
ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.set_title('Per-Disease Validation AUC')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(1, max(epochs))
fig.savefig(f"{GRAPHS_DIR}/03_per_disease_auc.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 4: Per-Disease F1 Curves
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for i, d in enumerate(DISEASE_NAMES):
    ax.plot(epochs, history['val_f1'][d], linewidth=2, color=colors[i], label=d)
ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Disease Validation F1 (Optimal Threshold)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(1, max(epochs))
fig.savefig(f"{GRAPHS_DIR}/04_per_disease_f1.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 5: Precision-Recall per Disease
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, d in enumerate(DISEASE_NAMES):
    ax = axes[i]
    ax.plot(epochs, history['val_precision'][d], color=colors[i], linewidth=2, linestyle='-', label='Precision')
    ax.plot(epochs, history['val_recall'][d], color=colors[i], linewidth=2, linestyle='--', label='Recall')
    ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
    ax.set_title(d, fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, max(epochs))
fig.suptitle('Per-Disease Precision vs Recall (Optimal Threshold)', fontsize=16, y=1.01)
fig.tight_layout()
fig.savefig(f"{GRAPHS_DIR}/05_precision_recall.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 6: Learning Rate Schedule (Actual)
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, history['lr'], 'purple', linewidth=2)
ax.axvline(WARMUP_EPOCHS, color='red', linestyle='--', alpha=0.5, label=f'Warmup end ({WARMUP_EPOCHS})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Actual Learning Rate Schedule')
ax.legend()
ax.grid(True, alpha=0.3)
ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
ax.set_xlim(1, max(epochs))
fig.savefig(f"{GRAPHS_DIR}/06_lr_schedule.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 7: Combined Dashboard
# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, history['train_loss'], 'b-', linewidth=1.5, label='Train')
ax1.plot(epochs, history['val_loss'], 'r-', linewidth=1.5, label='Val')
ax1.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, max(epochs))

# Macro F1
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, history['val_macro_f1_opt'], 'green', linewidth=1.5, label='Val F1 (opt)')
ax2.plot(epochs, history['train_macro_f1'], 'blue', linewidth=1, alpha=0.5, label='Train F1')
ax2.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
ax2.axhline(best_val_f1, color='green', linestyle=':', alpha=0.5)
ax2.set_title(f'Macro F1 (Best={best_val_f1:.4f})')
ax2.set_xlabel('Epoch')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, max(epochs))

# LR
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epochs, history['lr'], 'purple', linewidth=1.5)
ax3.axvline(WARMUP_EPOCHS, color='red', linestyle='--', alpha=0.5)
ax3.set_title('Learning Rate')
ax3.set_xlabel('Epoch')
ax3.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
ax3.grid(True, alpha=0.3)
ax3.set_xlim(1, max(epochs))

# Per-disease AUC
ax4 = fig.add_subplot(gs[1, :])
for i, d in enumerate(DISEASE_NAMES):
    ax4.plot(epochs, history['val_auc'][d], linewidth=2, color=colors[i], label=f'{d} (final={history["val_auc"][d][-1]:.3f})')
ax4.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
ax4.set_title('Per-Disease Validation AUC')
ax4.set_xlabel('Epoch')
ax4.legend(loc='lower right', fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(1, max(epochs))

# Per-disease F1
ax5 = fig.add_subplot(gs[2, :])
for i, d in enumerate(DISEASE_NAMES):
    ax5.plot(epochs, history['val_f1'][d], linewidth=2, color=colors[i], label=f'{d} (final={history["val_f1"][d][-1]:.3f})')
ax5.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
ax5.set_title('Per-Disease Validation F1 (Optimal Threshold)')
ax5.set_xlabel('Epoch')
ax5.legend(loc='lower right', fontsize=9)
ax5.grid(True, alpha=0.3)
ax5.set_xlim(1, max(epochs))

fig.suptitle(f'GLAAM-4X v4 Training Dashboard — {MODEL_NAME}', fontsize=16, y=1.01)
fig.savefig(f"{GRAPHS_DIR}/07_dashboard.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 8: Train vs Validation Loss
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss', alpha=0.8)
ax.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Validation Loss', alpha=0.8)
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('ASL Loss')
ax.set_title('Training vs Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(1, max(epochs))
fig.savefig(f"{GRAPHS_DIR}/08_train_val_loss.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 9: ROC Curves (per disease, from best-epoch validation)
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc as sk_auc

# Re-evaluate on validation set with best model to get final-epoch logits
best_ckpt_for_roc = torch.load(f"{RUN_DIR}/best_model.pth", map_location=device, weights_only=False)
model.load_state_dict(best_ckpt_for_roc['model_state_dict'])
model.eval()
roc_logits, roc_labels = evaluate(model, val_tune_loader)
roc_probs = 1 / (1 + np.exp(-roc_logits))

fig, ax = plt.subplots(figsize=(8, 7))
for i, d in enumerate(DISEASE_NAMES):
    y_true = roc_labels[:, i]
    y_prob = roc_probs[:, i]
    if len(np.unique(y_true)) > 1:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = sk_auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, color=colors[i], label=f'{d} (AUC={roc_auc:.3f})')
    else:
        ax.plot([], [], linewidth=2, color=colors[i], label=f'{d} (N/A — single class)')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Per-Disease ROC Curves (Best-Epoch Validation)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(0.0, 1.01)
fig.savefig(f"{GRAPHS_DIR}/09_roc_curves.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 10: Precision-Recall Curves (per disease, best-epoch val)
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 7))
for i, d in enumerate(DISEASE_NAMES):
    y_true = roc_labels[:, i]
    y_prob = roc_probs[:, i]
    if len(np.unique(y_true)) > 1:
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        ap = average_precision_score(y_true, y_prob)
        ax.plot(rec, prec, linewidth=2, color=colors[i], label=f'{d} (AP={ap:.3f})')
    else:
        ax.plot([], [], linewidth=2, color=colors[i], label=f'{d} (N/A — single class)')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Per-Disease Precision-Recall Curves (Best-Epoch Validation)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0.0, 1.01)
ax.set_ylim(0.0, 1.01)
fig.savefig(f"{GRAPHS_DIR}/10_pr_curves.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# Graph 11: Confusion Matrices (per disease, optimal threshold)
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
final_thresholds = best_ckpt_for_roc.get('optimal_thresholds', {d: 0.5 for d in DISEASE_NAMES})
for i, d in enumerate(DISEASE_NAMES):
    ax = axes[i]
    y_true = roc_labels[:, i].astype(int)
    y_prob = roc_probs[:, i]
    thr = final_thresholds.get(d, 0.5)
    y_pred = (y_prob >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # Build labels with both count and percentage
    cm_total = cm.sum()
    cm_pct = cm / cm_total * 100
    labels = np.array([[f'{cm[0,0]}\n({cm_pct[0,0]:.1f}%)', f'{cm[0,1]}\n({cm_pct[0,1]:.1f}%)'],
                       [f'{cm[1,0]}\n({cm_pct[1,0]:.1f}%)', f'{cm[1,1]}\n({cm_pct[1,1]:.1f}%)']])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    # Overlay count+percentage annotations
    for row in range(2):
        for col in range(2):
            ax.text(col, row, labels[row, col], ha='center', va='center',
                    fontsize=11, color='white' if cm[row, col] > cm.max() * 0.5 else 'black')
    ax.set_title(f'{d} (thr={thr:.2f})', fontsize=13)
fig.suptitle('Per-Disease Confusion Matrices (Best-Epoch Validation, Optimal Thresholds)', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f"{GRAPHS_DIR}/11_confusion_matrices.png")
plt.show()

print(f"\n{'='*60}")
print(f"📊 TRAINING GRAPHS SAVED")
print(f"{'='*60}")
print(f"Location: {GRAPHS_DIR}/")
print(f"Files:")
for f in sorted(os.listdir(GRAPHS_DIR)):
    print(f"  {f}")
print(f"{'='*60}")

# Section 13: Final Evaluation and Drive Results Summary

This section runs the final test evaluation and prints a summary. Test-set evaluation is performed only after training is complete to avoid test-set leakage: using test performance to tune hyperparameters would inflate reported results. The cell reports per-test-set macro F1 and per-disease AUC and F1, and confirms that all artifacts are persisted to Drive. After it runs, the Drive folder `checkpoints/glaam4x_unified_v4_asl_384/` contains `best_model.pth`, `latest_model.pth`, `best_model.safetensors`, `config.json`, `results.json`, `training_history.json`, and the `training_graphs/` directory.

In [ ]:
# ═══════════════════════════════════════════════════════
# Final Evaluation on Test Sets
# ═══════════════════════════════════════════════════════
print("=" * 60)
print(f"GLAAM-4X v4 — Final Test Evaluation")
print("=" * 60)

# Run test evaluation using the best checkpoint saved to Drive
results = run_test_evaluation(
    model,
    RUN_DIR,
    test_csv,
    test_loader,
    test_extended_csv=test_extended_csv if test_extended_csv.exists() else None
)

print(f"\n{'='*60}")
print(f"  ✅ ALL DONE!")
print(f"  Results saved to Drive:")
print(f"     {RUN_DIR}/")
print(f"     ├── best_model.pth          (full checkpoint, ~20MB)")
print(f"     ├── best_model.safetensors  (safe format, ~14MB)")
print(f"     ├── latest_model.pth")
print(f"     ├── results.json")
print(f"     └── config.json")
print(f"{'='*60}")

# Print summary from results.json
results_path = os.path.join(RUN_DIR, "results.json")
if os.path.exists(results_path):
    with open(results_path) as f:
        final_results = json.load(f)
    print(f"\n📊 Final Results Summary:")
    for test_name, test_data in final_results.get("test_results", {}).items():
        print(f"  {test_name} ({test_data['num_images']} images):")
        m = test_data["metrics"]
        print(f"    Macro F1: {m['macro_f1']:.4f}")
        for d in DISEASE_NAMES:
            dm = m[d]
            print(f"    {d:12s} AUC={dm['auc']:.4f} F1={dm['f1']:.4f}")
else:
    print(f"\n📊 Results not yet available at {results_path}")


# Section 14: Export Inference-Ready Artifacts

This section packages the trained model into a portable inference bundle so it can be deployed without re-running training. The bundle includes clean model weights (`model_weights.pth` and `best_model.safetensors`), per-disease optimal thresholds (`thresholds.json`), training metadata (`model_info.json`), a standalone inference script (`predict.py`), and usage instructions (`README.md`). A zip archive is also created for one-click download. The inference script can be run from the command line (`python predict.py --weights model_weights.pth --image fundus.jpg --thresholds thresholds.json`) or imported as a Python API, making the model usable in clinics, web services, mobile pipelines, or other cloud instances.

In [ ]:
import zipfile

# ═══════════════════════════════════════════════════════════════
# Export Inference-Ready Artifacts to Drive
# ═══════════════════════════════════════════════════════════════

INFERENCE_DIR = f"{DRIVE_BASE}/inference_package_{MODEL_NAME}"
os.makedirs(INFERENCE_DIR, exist_ok=True)

print(f"Exporting inference artifacts to: {INFERENCE_DIR}/")

# 1. Save clean model weights (state_dict only, no optimizer)
clean_weights_path = f"{INFERENCE_DIR}/model_weights.pth"
torch.save(model.state_dict(), clean_weights_path)
print(f"✅ Clean weights saved: {clean_weights_path}")

# 2. Save optimal thresholds
thresholds_path = f"{INFERENCE_DIR}/thresholds.json"
with open(thresholds_path, 'w') as f:
    json.dump(final_results['optimal_thresholds'], f, indent=2)
print(f"✅ Thresholds saved: {thresholds_path}")

# 3. Save model metadata
model_info = {
    "model_name": MODEL_NAME,
    "architecture": "GLAAM-4X",
    "backbone": "MobileNetV2",
    "num_classes": 4,
    "disease_names": DISEASE_NAMES,
    "img_size": IMG_SIZE,
    "dropout_rate": DROPOUT_RATE,
    "best_epoch": final_results.get('best_epoch', 0),
    "best_val_tune_f1": final_results.get('best_val_tune_f1', 0.0),
    "test_results": {
        k: {
            "num_images": v['num_images'],
            "macro_f1": v['metrics']['macro_f1']
        }
        for k, v in final_results.get('test_results', {}).items()
    },
    "training_config": {
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "epochs_trained": best_epoch,
        "warmup_epochs": WARMUP_EPOCHS,
        "asl_gamma_neg": ASL_GAMMA_NEG,
        "asl_gamma_pos": ASL_GAMMA_POS,
        "asl_clip": ASL_CLIP,
    }
}
model_info_path = f"{INFERENCE_DIR}/model_info.json"
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"✅ Model info saved: {model_info_path}")

# 4. Create standalone predict.py script
predict_script = '''#!/usr/bin/env python3
"""
Standalone inference script for GLAAM-4X v4
Load model, run prediction on single image or folder.
"""
import torch
import torch.nn as nn
import cv2
import numpy as np
import json
from pathlib import Path

DISEASE_NAMES = ['Cataract', 'DR', 'Glaucoma', 'Myopia']

class GLAAM4XClassifier(nn.Module):
    REORDER_IDX = [2, 0, 1, 3]
    def __init__(self, dropout_rate=0.3, pretrained=False):
        super().__init__()
        # NOTE: Update this import path if your project structure differs
        from models.glaam_4x import GLAAM_4X
        self.backbone = GLAAM_4X(pretrained=pretrained, dropout_rate=dropout_rate)
    def forward(self, x):
        out = self.backbone(x)
        logits = out['logits']
        return logits[:, self.REORDER_IDX]

def load_model(weights_path, device='cuda'):
    model = GLAAM4XClassifier(dropout_rate=0.3, pretrained=False)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def preprocess_image(img_path, img_size=384):
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    img = img.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = (img - mean) / std
    img = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)
    return img

def predict(model, img_path, thresholds=None, device='cuda', img_size=384):
    if thresholds is None:
        thresholds = {d: 0.5 for d in DISEASE_NAMES}
    img = preprocess_image(img_path, img_size).to(device)
    with torch.no_grad():
        logits = model(img)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    results = {}
    for i, disease in enumerate(DISEASE_NAMES):
        prob = float(probs[i])
        pred = int(prob >= thresholds.get(disease, 0.5))
        results[disease] = {"probability": round(prob, 4), "prediction": pred}
    return results

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--weights", required=True, help="Path to model_weights.pth")
    parser.add_argument("--image", required=True, help="Path to fundus image")
    parser.add_argument("--thresholds", default="thresholds.json", help="Path to thresholds.json")
    parser.add_argument("--device", default="cuda", help="cuda or cpu")
    parser.add_argument("--img_size", type=int, default=384)
    args = parser.parse_args()

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    model = load_model(args.weights, device)

    thresholds = {d: 0.5 for d in DISEASE_NAMES}
    if Path(args.thresholds).exists():
        with open(args.thresholds) as f:
            thresholds = json.load(f)

    results = predict(model, args.image, thresholds, device, args.img_size)
    print(json.dumps(results, indent=2))
'''

predict_path = f"{INFERENCE_DIR}/predict.py"
with open(predict_path, 'w') as f:
    f.write(predict_script)
print(f"✅ Inference script saved: {predict_path}")

# 5. Create README for the package
readme = f'''# GLAAM-4X v4 Inference Package

## Contents
- `model_weights.pth` — Clean model state dict (~14MB)
- `best_model.safetensors` — SafeTensors format (same weights)
- `thresholds.json` — Per-disease optimal thresholds
- `model_info.json` — Training metadata and config
- `predict.py` — Standalone inference script

## Quick Start

### Single Image Prediction
```bash
python predict.py \\
    --weights model_weights.pth \\
    --image path/to/fundus.jpg \\
    --thresholds thresholds.json
```

### Load in Python
```python
import torch
from predict import load_model, predict

model = load_model("model_weights.pth", device="cuda")
results = predict(model, "fundus.jpg", thresholds={"Cataract": 0.45, ...})
print(results)
# {{"Cataract": {{"probability": 0.9234, "prediction": 1}}, ...}}
```

## Model Info
- Architecture: GLAAM-4X (MobileNetV2 + 4 attention heads)
- Input size: {IMG_SIZE}x{IMG_SIZE}
- Diseases: {', '.join(DISEASE_NAMES)}
- Best Val F1: {final_results.get('best_val_tune_f1', 0):.4f}

## Thresholds
'''
for d in DISEASE_NAMES:
    thr = final_results.get('optimal_thresholds', {}).get(d, 0.5)
    readme += f"- {d}: {thr:.2f}\n"

readme_path = f"{INFERENCE_DIR}/README.md"
with open(readme_path, 'w') as f:
    f.write(readme)
print(f"✅ README saved: {readme_path}")

# 6. Copy best safetensors to inference dir
import shutil
src_safetensors = f"{RUN_DIR}/best_model.safetensors"
dst_safetensors = f"{INFERENCE_DIR}/best_model.safetensors"
if os.path.exists(src_safetensors):
    shutil.copy(src_safetensors, dst_safetensors)
    print(f"✅ SafeTensors copied to inference package")
else:
    print(f"⚠️  best_model.safetensors not found; skipping copy")

# 7. Create zip archive for easy download
zip_path = f"{DRIVE_BASE}/{MODEL_NAME}_inference_package.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ['model_weights.pth', 'best_model.safetensors', 'thresholds.json',
              'model_info.json', 'predict.py', 'README.md']:
        fp = f"{INFERENCE_DIR}/{f}"
        if os.path.exists(fp):
            zf.write(fp, f)
        elif f == 'best_model.safetensors':
            print(f"⚠️  {f} not present; continuing zip without it")
print(f"✅ Zip package created: {zip_path}")

print(f"\n{'='*60}")
print(f"📦 INFERENCE PACKAGE READY")
print(f"{'='*60}")
print(f"Location: {INFERENCE_DIR}/")
print(f"Zip:      {zip_path}")
print(f"\nFiles:")
for f in sorted(os.listdir(INFERENCE_DIR)):
    size = os.path.getsize(f"{INFERENCE_DIR}/{f}") / (1024*1024)
    print(f"  {f:30s} ({size:.1f} MB)")
print(f"{'='*60}")
